# Notebook 01b - Data Cleaning with Pandas (Member 1)
**HMDA 2024 project - Group 7.** This notebook: mount Drive → read the raw Parquet → apply the scope filter → clean with *Cleaning rules v1* → check the numbers → save `hmda_clean_v1.parquet`.

**How to use:** run the cells **from top to bottom**. This notebook uses Pandas only (no Spark). If Colab runs out of RAM: *Runtime → Restart session*, then run again from the first cell.

## Step 1. Mount Google Drive and copy the raw data to Colab
The folder `hmda/hmda_parquet` must be in your **My Drive** (if it was shared with you, use Add shortcut to Drive first).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
SRC = "/content/drive/MyDrive/hmda/hmda_parquet"   # raw Parquet on Drive
LOCAL = "/content/hmda_parquet"                      # local copy on Colab disk (faster to read)
assert os.path.exists(SRC), "hmda_parquet folder not found on Drive - check the shortcut / folder name"
if not os.path.exists(LOCAL):
    !cp -r "{SRC}" "{LOCAL}"
print("OK. Number of files in the folder:", len(os.listdir(LOCAL)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK. Number of files in the folder: 72


## Step 2. Import libraries

In [4]:
import json, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
print("pandas", pd.__version__, "| pyarrow", pa.__version__)

pandas 2.2.3 | pyarrow 23.0.1


## Step 3. Configuration: scope, columns to read, band edges
This is *Cleaning rules v1* written as code. Nothing to edit in this cell.

In [5]:
SCOPE_VALUES = {                      # column -> required string value(s)
    "action_taken": ["1", "2", "3"],
    "occupancy_type": ["1"],
    "loan_purpose": ["1"],
    "lien_status": ["1"],
    "derived_dwelling_category": ["Single Family (1-4 Units):Site-Built"],
    "reverse_mortgage": ["2"],
    "open_end_line_of_credit": ["2"],
    "business_or_commercial_purpose": ["2"],
}
NUMERIC_COLS = ["loan_amount", "income", "combined_loan_to_value_ratio",
                "property_value", "loan_term"]
KEEP_COLS = ["action_taken", "state_code", "derived_msa_md", "loan_type", "lei",
             "debt_to_income_ratio", "derived_race", "derived_ethnicity",
             "derived_sex", "applicant_age"] + NUMERIC_COLS
READ_COLS = sorted(set(KEEP_COLS) | set(SCOPE_VALUES))
CATEGORY_COLS = ["action_taken", "state_code", "derived_msa_md", "loan_type", "lei",
                 "debt_to_income_ratio", "derived_race", "derived_ethnicity",
                 "derived_sex", "applicant_age"]

INT32_MAX = 2147483647
DTI_ORDER = ["<20", "20-<30", "30-<36", "36-<40", "40-<45", "45-<50", "50-60", ">60", "Missing"]
DTI_TEXT = {"<20%": "<20", "20%-<30%": "20-<30", "30%-<36%": "30-<36",
            "50%-60%": "50-60", ">60%": ">60"}

REGIONS = {
    "Northeast": "CT ME MA NH RI VT NJ NY PA",
    "Midwest": "IL IN MI OH WI IA KS MN MO NE ND SD",
    "South": "DE DC FL GA MD NC SC VA WV AL KY MS TN AR LA OK TX",
    "West": "AZ CO ID MT NV NM UT WY AK CA HI OR WA",
    "Territory": "PR VI GU AS MP",
}
STATE_TO_REGION = {s: r for r, states in REGIONS.items() for s in states.split()}

## Step 4. Helper functions: DTI grouping and banding

In [6]:
def dti_to_band(v):
    """Map one raw DTI value (band text or exact integer 36-49) to an ordered band."""
    v = str(v).strip()
    if v in DTI_TEXT:
        return DTI_TEXT[v]
    try:
        x = float(v)
    except ValueError:
        return "Missing"                      # 'NA', 'Exempt' or anything unparseable
    if x < 20: return "<20"
    if x < 30: return "20-<30"
    if x < 36: return "30-<36"
    if x < 40: return "36-<40"
    if x < 45: return "40-<45"
    if x < 50: return "45-<50"
    if x <= 60: return "50-60"
    return ">60"


def to_band(x, edges, labels):
    """Right-open bins [a, b) with a trailing 'Missing' category."""
    b = pd.cut(x, bins=edges, labels=labels, right=False)
    b = b.cat.add_categories("Missing").fillna("Missing")
    return b.cat.reorder_categories(list(labels) + ["Missing"], ordered=True)

## Step 5. Cleaning function `clean()`
Every rule R3-R12 lives in this function. Invalid values are **flagged and set to null**; records are not dropped.

In [7]:
def clean(df):
    """df: in-scope rows with READ_COLS as strings. Returns (clean_df, log)."""
    log = {"rows_in_scope": int(len(df))}

    # R12: exact duplicates over the columns we keep (upper bound; not removed)
    log["exact_duplicate_rows_on_kept_columns"] = int(df[KEEP_COLS].duplicated().sum())

    # R3: special codes -> missing
    log["special_NA_or_Exempt"] = {c: int(df[c].isin(["NA", "Exempt"]).sum()) for c in NUMERIC_COLS}
    for c in NUMERIC_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    log["missing_after_numeric_conversion"] = {c: int(df[c].isna().sum()) for c in NUMERIC_COLS}
    log["unparseable_other_than_NA_Exempt"] = {
        c: log["missing_after_numeric_conversion"][c] - log["special_NA_or_Exempt"][c]
        for c in NUMERIC_COLS}

    # R4-R7: invalid values -> flag + missing
    f = pd.DataFrame(index=df.index)
    f["income_invalid"] = (df["income"] <= 0)
    f["cltv_invalid"] = (df["combined_loan_to_value_ratio"] > 200) | (df["combined_loan_to_value_ratio"] < 5)
    f["property_value_invalid"] = (df["property_value"] == INT32_MAX)
    f["loan_term_invalid"] = (df["loan_term"] < 12) | (df["loan_term"] > 480)
    log["loan_amount_non_positive"] = int((df["loan_amount"] <= 0).sum())
    df.loc[f["income_invalid"], "income"] = np.nan
    df.loc[f["cltv_invalid"], "combined_loan_to_value_ratio"] = np.nan
    df.loc[f["property_value_invalid"], "property_value"] = np.nan
    df.loc[f["loan_term_invalid"], "loan_term"] = np.nan
    log["invalid_flagged_and_nulled"] = {c: int(f[c].sum()) for c in f.columns}

    out = pd.DataFrame(index=df.index)
    out["action_taken"] = pd.to_numeric(df["action_taken"]).astype("int8")
    out["denied"] = (out["action_taken"] == 3).astype("int8")              # R2
    out["state_code"] = df["state_code"].astype("category")
    out["census_region"] = (df["state_code"].astype("string").map(STATE_TO_REGION)
                            .fillna("Unknown").astype("category"))
    out["derived_msa_md"] = df["derived_msa_md"].astype("category")
    non_metro = df["derived_msa_md"].astype("string") == "99999"            # R9
    out["metro_flag"] = (~non_metro).astype("int8")
    out["msa_md_clean"] = (df["derived_msa_md"].astype("string").mask(non_metro, "Non-metro")
                           .astype("category"))
    out["loan_type"] = df["loan_type"].astype("category")
    out["lei"] = df["lei"].astype("category")

    for c in NUMERIC_COLS:
        out[c] = df[c]
    out["loan_to_income"] = df["loan_amount"] / (df["income"] * 1000)       # R10 (NaN if income missing)

    # R8: DTI bands via the (few) unique raw values
    raw_dti = df["debt_to_income_ratio"].astype("category")
    mapping = {k: dti_to_band(k) for k in raw_dti.cat.categories}
    log["dti_raw_values_mapped_to_Missing"] = sorted(
        k for k, v in mapping.items() if v == "Missing")
    out["dti_band"] = pd.Categorical(raw_dti.map(mapping).astype(str), categories=DTI_ORDER, ordered=True)

    out["income_band"] = to_band(out["income"], [0, 50, 75, 100, 150, 250, np.inf],
                                 ["<50", "50-<75", "75-<100", "100-<150", "150-<250", "250+"])
    out["loan_amount_band"] = to_band(out["loan_amount"], [0, 150e3, 250e3, 350e3, 500e3, 1e6, np.inf],
                                      ["<150k", "150-<250k", "250-<350k", "350-<500k", "500k-<1M", "1M+"])
    out["cltv_band"] = to_band(out["combined_loan_to_value_ratio"], [0, 60, 80, 90, 97, np.inf],
                               ["<60", "60-<80", "80-<90", "90-<97", "97+"])

    for c in f.columns:                                                     # flags
        out[c] = f[c].astype("int8")
    out["income_gt5000"] = (out["income"] > 5000).astype("int8")            # R11
    out["lti_gt10"] = (out["loan_to_income"] > 10).astype("int8")           # R10
    log["kept_but_flagged"] = {"income_gt5000": int(out["income_gt5000"].sum()),
                               "lti_gt10": int(out["lti_gt10"].sum())}

    # validation-only demographics (never use as model features)
    out["val_race"] = df["derived_race"].astype("category")
    out["val_ethnicity"] = df["derived_ethnicity"].astype("category")
    out["val_sex"] = df["derived_sex"].astype("category")
    out["val_age"] = df["applicant_age"].astype("category")

    log["rows_out"] = int(len(out))
    log["denied_rows"] = int(out["denied"].sum())
    log["denied_rate"] = round(float(out["denied"].mean()), 6)
    log["missing_by_column_out"] = {c: int(out[c].isna().sum()) for c in
                                    ["income", "combined_loan_to_value_ratio", "property_value",
                                     "loan_term", "loan_to_income"]}
    log["dti_band_counts"] = {k: int(v) for k, v in out["dti_band"].value_counts().items()}
    return out.reset_index(drop=True), log

## Step 6. Read only the in-scope data
Only the needed columns and the rows that satisfy the scope are loaded (not all 12 million rows), and the number of records after each filter is counted.

In [8]:
def read_scope(parquet_dir):
    """Read only the in-scope rows and needed columns; also return step-by-step row counts."""
    import pyarrow.dataset as ds
    dset = ds.dataset(parquet_dir, format="parquet")
    steps = {"total_records": int(dset.count_rows())}
    expr = None
    for col, vals in SCOPE_VALUES.items():
        e = ds.field(col).isin(vals)
        expr = e if expr is None else (expr & e)
        steps[f"after_filter_{col}"] = int(dset.count_rows(filter=expr))
    tbl = dset.to_table(columns=READ_COLS, filter=expr)
    df = tbl.to_pandas(categories=[c for c in CATEGORY_COLS], self_destruct=True)
    return df, steps

## Step 7. Run: read → clean
Takes a few minutes. Result: the cleaned table `out` and `log` (counts for each step).

In [9]:
t0 = time.perf_counter()
df, steps = read_scope(LOCAL)
print(f"Read {len(df):,} rows in {time.perf_counter()-t0:.1f}s")
print(json.dumps(steps, indent=2))

Read 3,140,857 rows in 52.5s
{
  "total_records": 12259199,
  "after_filter_action_taken": 8661779,
  "after_filter_occupancy_type": 7792568,
  "after_filter_loan_purpose": 3811298,
  "after_filter_lien_status": 3641962,
  "after_filter_derived_dwelling_category": 3237835,
  "after_filter_reverse_mortgage": 3168105,
  "after_filter_open_end_line_of_credit": 3145390,
  "after_filter_business_or_commercial_purpose": 3140857
}


In [10]:
t0 = time.perf_counter()
out, log = clean(df)
log["scope_step_counts"] = steps
del df                                   # free RAM
print(f"Cleaned {len(out):,} rows in {time.perf_counter()-t0:.1f}s")

Cleaned 3,140,857 rows in 26.3s


## Step 8. Check the numbers against the earlier measurements
Every line must show **OK**. If any line shows **MISMATCH**, stop and do not save.

In [11]:
expected = {
    "total_records":                    (steps["total_records"], 12259199),
    "records with action_taken 1-3":    (steps["after_filter_action_taken"], 8661779),
    "rows in scope":                    (log["rows_in_scope"], 3140857),
    "denied rows":                      (log["denied_rows"], 276604),
    "NA/Exempt income":                 (log["special_NA_or_Exempt"]["income"], 21366),
    "NA/Exempt CLTV":                   (log["special_NA_or_Exempt"]["combined_loan_to_value_ratio"], 96968),
    "NA/Exempt property_value":         (log["special_NA_or_Exempt"]["property_value"], 29439),
    "NA/Exempt loan_term":              (log["special_NA_or_Exempt"]["loan_term"], 6160),
    "income <= 0 (invalid)":            (log["invalid_flagged_and_nulled"]["income_invalid"], 16942),
    "CLTV >200 or <5 (invalid)":        (log["invalid_flagged_and_nulled"]["cltv_invalid"], 371 + 288),
    "property_value = 2^31-1 (invalid)":(log["invalid_flagged_and_nulled"]["property_value_invalid"], 2),
    "loan_term <12 or >480 (invalid)":  (log["invalid_flagged_and_nulled"]["loan_term_invalid"], 1001 + 4),
    "income > 5000 (flag)":             (log["kept_but_flagged"]["income_gt5000"], 857),
    "loan_to_income > 10 (flag)":       (log["kept_but_flagged"]["lti_gt10"], 11937),
    "DTI band <20":    (log["dti_band_counts"]["<20"], 113535),
    "DTI band 20-<30": (log["dti_band_counts"]["20-<30"], 390684),
    "DTI band 30-<36": (log["dti_band_counts"]["30-<36"], 449579),
    "DTI band 36-<40": (log["dti_band_counts"]["36-<40"], 401994),
    "DTI band 40-<45": (log["dti_band_counts"]["40-<45"], 628198),
    "DTI band 45-<50": (log["dti_band_counts"]["45-<50"], 710151),
    "DTI band 50-60":  (log["dti_band_counts"]["50-60"], 338631),
    "DTI band >60":    (log["dti_band_counts"][">60"], 82947),
    "DTI band Missing":(log["dti_band_counts"]["Missing"], 25138),
    "unparseable (should be 0)":        (sum(log["unparseable_other_than_NA_Exempt"].values()), 0),
}
bad = 0
for name, (got, exp) in expected.items():
    ok = got == exp
    bad += (not ok)
    print(f"{'OK       ' if ok else 'MISMATCH '} {name:36s} got={got:>10,}  expected={exp:>10,}")
print("\n=> ALL MATCH" if bad == 0 else f"\n=> {bad} metrics MISMATCH - do not save the results")

OK        total_records                        got=12,259,199  expected=12,259,199
OK        records with action_taken 1-3        got= 8,661,779  expected= 8,661,779
OK        rows in scope                        got= 3,140,857  expected= 3,140,857
OK        denied rows                          got=   276,604  expected=   276,604
OK        NA/Exempt income                     got=    21,366  expected=    21,366
OK        NA/Exempt CLTV                       got=    96,968  expected=    96,968
OK        NA/Exempt property_value             got=    29,439  expected=    29,439
OK        NA/Exempt loan_term                  got=     6,160  expected=     6,160
OK        income <= 0 (invalid)                got=    16,942  expected=    16,942
OK        CLTV >200 or <5 (invalid)            got=       659  expected=       659
OK        property_value = 2^31-1 (invalid)    got=         2  expected=         2
OK        loan_term <12 or >480 (invalid)      got=     1,005  expected=     1,005
OK  

## Step 9. Save `hmda_clean_v1` and the log, then copy to Drive
Run only when Step 8 reports **ALL MATCH**.

In [12]:
OUT_LOCAL = "/content/hmda_clean_v1"
OUT_DRIVE = "/content/drive/MyDrive/hmda/hmda_clean_v1"
os.makedirs(OUT_LOCAL, exist_ok=True)
out.to_parquet(f"{OUT_LOCAL}/hmda_clean_v1.parquet", index=False)
with open(f"{OUT_LOCAL}/cleaning_log.json", "w") as fh:
    json.dump(log, fh, indent=2)
os.makedirs(OUT_DRIVE, exist_ok=True)
!cp "{OUT_LOCAL}/hmda_clean_v1.parquet" "{OUT_DRIVE}/"
!cp "{OUT_LOCAL}/cleaning_log.json" "{OUT_DRIVE}/"
!ls -lh "{OUT_DRIVE}"

total 46M
-rw------- 1 root root 1.8K Nov 20 16:43 cleaning_log.json
-rw------- 1 root root  46M Nov 20 16:43 hmda_clean_v1.parquet


## Step 10. Quick look at the result

In [13]:
display(out.head())
print(out.shape)
print(f"Memory: {out.memory_usage(deep=True).sum()/1e9:.2f} GB")
out.dtypes

,action_taken,denied,state_code,census_region,derived_msa_md,metro_flag,msa_md_clean,loan_type,lei,loan_amount,...,income_invalid,cltv_invalid,property_value_invalid,loan_term_invalid,income_gt5000,lti_gt10,val_race,val_ethnicity,val_sex,val_age
0,1,0,TX,South,19124,1,19124,3,254900HA4DQWAE0W3342,335000,...,0,0,0,0,0,0,Black or African American,Hispanic or Latino,Male,35-44
1,1,0,OK,South,36420,1,36420,2,254900HA4DQWAE0W3342,265000,...,0,0,0,0,0,0,White,Not Hispanic or Latino,Female,45-54
2,1,0,TX,South,15180,1,15180,1,254900HA4DQWAE0W3342,215000,...,0,0,0,0,0,0,Race Not Available,Hispanic or Latino,Male,35-44
3,1,0,IN,Midwest,29414,1,29414,1,254900HA4DQWAE0W3342,315000,...,0,0,0,0,0,0,White,Not Hispanic or Latino,Joint,25-34
4,1,0,TX,South,12420,1,12420,1,254900HA4DQWAE0W3342,365000,...,0,0,0,0,0,0,White,Not Hispanic or Latino,Joint,25-34


(3140857, 29)
Memory: 0.23 GB


,0
action_taken,int8
denied,int8
state_code,category
census_region,category
derived_msa_md,category
metro_flag,int8
msa_md_clean,category
loan_type,category
lei,category
loan_amount,int64


In [14]:
# The val_* columns are for after-the-fact validation only, NEVER features.
# Whitelist of columns allowed as model features (for Member 3):
FEATURES_ALLOWED = ["income", "loan_amount", "combined_loan_to_value_ratio", "property_value", "loan_term",
                    "loan_to_income", "dti_band", "income_band", "loan_amount_band", "cltv_band",
                    "loan_type", "state_code", "census_region", "metro_flag"]
assert not any(c.startswith("val_") for c in FEATURES_ALLOWED)
print("val_* columns (validation only):", [c for c in out.columns if c.startswith("val_")])

val_* columns (validation only): ['val_race', 'val_ethnicity', 'val_sex', 'val_age']
